# 04 — Model layer

Notebook 03 produced **`PreprocessedDataset`** — a clean `(X, y)` ready to train on. This notebook builds the **model layer**: a typed `BaseRegressor` ABC, a `ModelFactory`, and three concrete models that exercise the abstraction's extremes.

## Architecture

```text
BaseRegressor[P]              ABC, generic over typed params
    fit(X, y) -> Self
    predict(X) -> pd.Series   index aligned to X.index
    params: P                 typed, frozen, JSON-serialisable
    save(dir) / load(dir)     dispatched via ModelKind on disk

BaseModelParams               ABC, frozen
    kind: ModelKind           drives factory + persistence dispatch

ModelKind (StrEnum)
    .params_type()            kind → params dataclass
    .model_class()            kind → BaseRegressor subclass
```

## What each concrete model exercises

| Model | Why it's here |
|---|---|
| `MeanBaselineRegressor` | Sanity baseline. Anything worse than this in NB 06 has a bug. No scaling, no hyperparameters worth tuning — exercises the contract at its simplest. |
| `RandomForestRegressor` | Realistic baseline. Tree-based → no scaling needed. Exercises the abstraction with a non-trivial fitted state (`joblib`-pickled forest). |
| `LinearRegressor` | Exercises the **in-wrapper scaling contract**. Fits a `StandardScaler` on the training fold + an OLS regression. Future NN wrappers will use the same pattern. |

The architectural decision (carried over from NB 03):

> **Scaling lives in the model wrapper, not the Preprocessor.** RF doesn't need it; Linear does. Each wrapper owns its own scaler so cross-fold leakage is impossible by construction — the `Preprocessor` stays stateless, and the train fold is the *only* data that reaches `.fit(...)`.


In [ ]:
# ============================================================
# Colab bootstrap (no-op when run locally).
# ------------------------------------------------------------
# First-time setup on Colab:
#   1. Create a GitHub Personal Access Token (PAT) at
#      https://github.com/settings/tokens with `repo` scope.
#      The repository is private, so the clone needs this token
#      (or an SSH key Colab knows about, which is more fiddly).
#   2. Add the token under Tools → Secrets in Colab with name
#      `GITHUB_PAT` and toggle "Notebook access" on.
#   3. Sign in with a Google account that has BigQuery read access
#      to `solar-irradiation-estimation` when prompted.
# ============================================================
import os
import sys

if "google.colab" in sys.modules:
    REPO = "Marconi-Lab/Solar_irradiation"
    BRANCH = "jm/add_model"

    if not os.path.exists("/content/Solar_irradiation/.git"):
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_PAT")
            clone_url = f"https://{token}@github.com/{REPO}.git"
            print("Cloning with Colab secret 'GITHUB_PAT'.")
        except Exception:
            clone_url = f"git@github.com:{REPO}.git"
            print(
                "Colab secret 'GITHUB_PAT' not set — trying SSH. If the "
                "clone fails, follow the PAT setup steps above and re-run."
            )
        !git clone -q -b {BRANCH} {clone_url} /content/Solar_irradiation

    %cd /content/Solar_irradiation
    !pip install -q -e . 2>&1 | tail -3

    from google.colab import auth
    auth.authenticate_user()
    !gcloud config set project solar-irradiation-estimation 2>/dev/null
    print("Colab setup complete.")

### Inputs, outputs, and prerequisites

| | |
|---|---|
| **Inputs** | The canonical `TrainingDataset` snapshot from NB 02 at `data/training_snapshots/susse_training_demo_v1-2024/`. The same v1 `FeatureSpec` from NB 03 is re-applied inline below, so this notebook is self-contained. |
| **Outputs** | Three on-disk model directories at `data/models/_nb04_{mean_baseline,random_forest,linear}/`, each holding `model_kind.txt`, `params.json`, and `state.joblib`. |
| **Prereqs** | None for offline use — the snapshot loader works without GCP or W&B. No environment variables required. |
| **Consumed by** | NB 05's `Trainer` reuses the exact same factory + fit/predict/score/save flow, plus a `TrainedBundle` that pairs each fitted model with its `FeatureSpec` for inference. NB 06 replaces the inline station-LOSO below with a proper splitter abstraction. |

## 0 — Setup


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

src_path = (Path.cwd() / "../../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("Python", sys.version.split()[0])


## 1 — Load the preprocessed dataset from NB 03

This loads the canonical v1 snapshot (built by NB 02), applies the v1 `FeatureSpec` (defined inline below — same as NB 03), and produces a `PreprocessedDataset` with `X()` and `y()` slices ready for training.


In [ ]:
from susse.datasets import load_snapshot
from susse.preprocessing import (
    ClearSkyIndexFeature,
    CyclicalDayOfYearFeature,
    FeatureSpec,
    Preprocessor,
)

SNAPSHOT_DIR = (Path.cwd() / "../../data/training_snapshots/susse_training_demo_v1-2024").resolve()
dataset = load_snapshot(SNAPSHOT_DIR)
print(f"Loaded {dataset.manifest.name} {dataset.manifest.version}: "
      f"{dataset.manifest.n_rows} rows x {dataset.manifest.n_cols} cols")

spec = FeatureSpec(
    target_column="y_ghi_kwh_m2_day",
    feature_columns=(
        "nasa_aod_550",
        "nasa_precipitable_water",
        "nasa_cloud_amount",
        "nasa_clearness_index",
        "sat_ghi_nasa_kwh_m2_day",
        "sat_ghi_cams_kwh_m2_day",
    ),
    derived_features=(
        ClearSkyIndexFeature(
            ghi_column="sat_ghi_cams_kwh_m2_day",
            ghi_clear_column="cams_ghi_clear",
            output_column="kt_cams",
        ),
        CyclicalDayOfYearFeature(),
    ),
)

processed = Preprocessor(spec).apply(dataset)
print(f"Processed: {processed.n_rows} rows x {processed.n_features} features")
print(f"Feature columns: {processed.feature_columns}")

## 2 - Station-leave-one-out split

A random row-level split would let the model see rows from every station during training, then evaluate it on more rows from the *same* stations - the model can memorise per-site mean shifts and look better than it really is.

Real deployment is the opposite: we run the model on points where we have **no ground truth at all**. The honest stand-in is to hold out an entire station, train on all the others, and evaluate on the held-out one. The proper splitter abstraction (random / temporal / station-LOSO / spatial-block) is decided in NB 06; here we inline a simple "pick the station with the most 2024 days, hold it out" rule for the demo.


In [ ]:
import pandas as pd
import numpy as np

# Pick the held-out station automatically: whichever has the most rows
# in 2024. Keeps the demo robust to changes in which stations the
# warehouse covers, and gives the densest possible held-out time series
# for the annual-profile plot below.
rows_per_station = (
    processed.df.groupby("location").size().sort_values(ascending=False)
)
print("Rows per station (2024):")
print(rows_per_station.head(10))
print()

held_out_station = str(rows_per_station.idxmax())
print(f"Held out: {held_out_station!r} ({rows_per_station.iloc[0]} rows)")

train_mask = processed.df["location"] != held_out_station
test_mask = ~train_mask

X_train = processed.X().loc[train_mask]
y_train = processed.y().loc[train_mask]
X_test = processed.X().loc[test_mask]
y_test = processed.y().loc[test_mask]

print(f"Train: {len(X_train):,} rows from "
      f"{processed.df.loc[train_mask, 'location'].nunique()} stations")
print(f"Test:  {len(X_test):,} rows from {held_out_station} only")
print(f"y_train mean = {y_train.mean():.3f} kWh/m2/day "
      f"(std = {y_train.std():.3f})")

## 3 — Train all three models via `ModelFactory`

Each call follows the same shape: `params → ModelFactory.create(params) → .fit(X, y)`. The typed params dataclass is the only API surface; no `model_type="random_forest"` strings, no `dict(params)` to misspell.


### Hyperparameter choices in v1

The values below are deliberate starting points, not tuned — proper tuning belongs in NB 06's evaluation loop.

- **`MeanBaselineParams()`** — no hyperparameters worth tuning; predicts the train mean for every test row. Anything worse than this in NB 06 has a bug.
- **`RandomForestParams(n_estimators=200, max_depth=12, random_state=42)`** — 200 trees keep predictions stable while `.fit()` stays fast on a 6k-row training set; `max_depth=12` is a heuristic ceiling against overfit on this dataset size; `random_state=42` makes the notebook reproducible.
- **`LinearParams(with_scaling=True)`** — OLS with a `StandardScaler` fitted *only* on the train fold. Scaling matters here because feature magnitudes differ widely (`kt_cams ~ 0–1` vs `sat_ghi ~ 0–10`); flipping to `with_scaling=False` would let high-magnitude features dominate the fit.

The full inventory of available models is `list(ModelKind)` — currently `['mean_baseline', 'random_forest', 'linear']`. Adding a fourth (e.g. XGBoost) is one new enum member, one params dataclass, and one regressor subclass; the table at the top of this notebook walks through the recipe.

In [ ]:
from susse.models import (
    LinearParams, MeanBaselineParams, ModelFactory, RandomForestParams,
)

mean_model = ModelFactory.create(MeanBaselineParams()).fit(X_train, y_train)
rf_model = ModelFactory.create(
    RandomForestParams(n_estimators=200, max_depth=12, random_state=42)
).fit(X_train, y_train)
linear_model = ModelFactory.create(
    LinearParams(with_scaling=True)
).fit(X_train, y_train)

print(f"Trained: {type(mean_model).__name__}, "
      f"{type(rf_model).__name__}, "
      f"{type(linear_model).__name__}")


## 4 - Score on the held-out station

These numbers reflect *generalisation to a station the model has never seen*. The mean-baseline number is a useful sanity check (it predicts the overall train mean for every day, blind to weather); the satellite estimates (NASA, CAMS) are what the model is meant to *correct*; RF and Linear are the bias-corrected outputs. The visual comparison below makes systematic biases easier to read than the table alone.


In [ ]:
def _score(
    y_true: pd.Series,
    y_pred: pd.Series,
    *,
    mask: pd.Series | None = None,
) -> tuple[float, float, float]:
    if mask is not None:
        y_true = y_true[mask]
        y_pred = y_pred[mask]
    err = y_pred - y_true
    mae = err.abs().mean()
    rmse = np.sqrt((err ** 2).mean())
    ss_res = (err ** 2).sum()
    ss_tot = ((y_true - y_true.mean()) ** 2).sum()
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    return mae, rmse, r2


nasa_pred = processed.df.loc[test_mask, "sat_ghi_nasa_kwh_m2_day"]
cams_pred = processed.df.loc[test_mask, "sat_ghi_cams_kwh_m2_day"]

# Score every series only on rows where it (and the target) are non-null;
# some grid cells / dates can have NaN sat_ghi.
rows = []
for name, preds in [
    ("NASA satellite", nasa_pred),
    ("CAMS satellite", cams_pred),
    ("MeanBaseline",   mean_model.predict(X_test)),
    ("Linear",         linear_model.predict(X_test)),
    ("RandomForest",   rf_model.predict(X_test)),
]:
    mask = preds.notna() & y_test.notna()
    mae, rmse, r2 = _score(y_test, preds, mask=mask)
    rows.append({"model": name, "mae": mae, "rmse": rmse, "r2": r2,
                 "n_rows": int(mask.sum())})

scores = pd.DataFrame(rows).set_index("model")
scores

### Reading the score table

- **MeanBaseline**'s negative R² is the right sign that the LOSO holdout is doing real work — predicting the train-set mean is *worse* than a hypothetical predictor that already knew the test station's mean, because seasonality and station-specific shift differ. R² < 0 here means "any honest information about today's weather improves on this."
- **NASA / CAMS satellite** are the *raw inputs* the model is asked to bias-correct. Their MAE around 1.0 kWh/m²/day is the bar both ML models have to beat.
- **Linear** halves the satellite MAE (0.56) — most of the bias is linearly removable from the seven features we provided.
- **RandomForest** halves it again (0.37, R² = 0.86) — the non-linear feature interactions carry real signal.

Two cautions before reading these as final: the held-out station (`tororo`) may not be representative of every deployment region — NB 06 will fold across all stations — and station-specific biases that *are* learnable across other stations get rewarded harder by LOSO than by random splits, so this is closer to a deployment estimate than a fitted-data estimate.

## 5 - Annual profile at the held-out station

Daily GHI through the year for the held-out station: the **target line** is what the model is trying to match, NASA / CAMS are the **inputs** the model is asked to bias-correct, and the model lines (RF, Linear) are the **outputs** to compare. The mean baseline is shown as a horizontal reference - it has no information beyond the train mean.

A 7-day rolling mean below the daily plot smooths out single-day cloudiness and makes systematic biases (which is what we're trying to remove) easier to read.


In [ ]:
profile = processed.df.loc[test_mask, ["date", "y_ghi_kwh_m2_day"]].copy()
profile["NASA satellite"] = processed.df.loc[test_mask, "sat_ghi_nasa_kwh_m2_day"]
profile["CAMS satellite"] = processed.df.loc[test_mask, "sat_ghi_cams_kwh_m2_day"]
profile["RandomForest"] = rf_model.predict(X_test)
profile["Linear"] = linear_model.predict(X_test)
profile["MeanBaseline"] = float(y_train.mean())  # constant
profile = (
    profile.assign(date=pd.to_datetime(profile["date"]))
           .set_index("date")
           .sort_index()
)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

# Top: daily series.
axes[0].plot(profile.index, profile["y_ghi_kwh_m2_day"], color="black",     lw=1.6, alpha=0.9, label="ground truth")
axes[0].plot(profile.index, profile["NASA satellite"],   color="tab:blue",  lw=1.0, alpha=0.6, label="NASA satellite")
axes[0].plot(profile.index, profile["CAMS satellite"],   color="tab:orange", lw=1.0, alpha=0.6, label="CAMS satellite")
axes[0].plot(profile.index, profile["RandomForest"],     color="tab:green", lw=1.4, alpha=0.85, label="RandomForest")
axes[0].plot(profile.index, profile["Linear"],           color="tab:purple", lw=1.2, alpha=0.7, ls="--", label="Linear")
axes[0].plot(profile.index, profile["MeanBaseline"],     color="gray",      lw=0.8, alpha=0.6, ls=":", label="MeanBaseline")
axes[0].set_ylabel("GHI (kWh/m^2/day)")
axes[0].set_title(f"Annual GHI profile at held-out station: {held_out_station}")
axes[0].legend(loc="lower center", ncol=6, fontsize=9, frameon=False)
axes[0].grid(alpha=0.25)

# Bottom: 7-day rolling mean.
rolling = profile.rolling("7D", min_periods=3).mean()
axes[1].plot(rolling.index, rolling["y_ghi_kwh_m2_day"], color="black",     lw=1.6,             label="ground truth")
axes[1].plot(rolling.index, rolling["NASA satellite"],   color="tab:blue",  lw=1.2, alpha=0.7,  label="NASA satellite")
axes[1].plot(rolling.index, rolling["CAMS satellite"],   color="tab:orange", lw=1.2, alpha=0.7, label="CAMS satellite")
axes[1].plot(rolling.index, rolling["RandomForest"],     color="tab:green", lw=1.6,             label="RandomForest")
axes[1].plot(rolling.index, rolling["Linear"],           color="tab:purple", lw=1.4, ls="--",   label="Linear")
axes[1].plot(rolling.index, rolling["MeanBaseline"],     color="gray",      lw=1.0, ls=":",     label="MeanBaseline")
axes[1].set_ylabel("GHI (kWh/m^2/day) - 7-day mean")
axes[1].set_xlabel("date (2024)")
axes[1].grid(alpha=0.25)

plt.tight_layout()

## 5 — Save / load roundtrip

Each model persists into a directory holding `model_kind.txt`, `params.json`, and `state.joblib`. The package-level `load_regressor(dir)` reads the kind tag and dispatches to the right concrete class — no caller-side knowledge of which model was trained.


In [ ]:
from susse.models import load_regressor

MODELS_ROOT = (Path.cwd() / "../../data/models").resolve()
MODELS_ROOT.mkdir(parents=True, exist_ok=True)

for name, model in [
    ("mean_baseline", mean_model),
    ("random_forest", rf_model),
    ("linear", linear_model),
]:
    out = MODELS_ROOT / f"_nb04_{name}"
    model.save(out)
    restored = load_regressor(out)
    # Tolerance accommodates ~1-ulp drift from joblib pickling/unpickling
    # of linear coefficients — well below any meaningful prediction change.
    np.testing.assert_allclose(
        restored.predict(X_test).values, model.predict(X_test).values,
        rtol=1e-10, atol=1e-10,
    )
    print(f"{name}: roundtrip OK — restored as {type(restored).__name__}")


The persistence format is stable and human-readable — `cat model_kind.txt` and `cat params.json` are sensible operations. State (the fitted forest, the scaler + linear coefficients) lives in a single joblib blob.


In [ ]:
example_dir = MODELS_ROOT / "_nb04_random_forest"
print("Files in", example_dir.name, ":")
for f in sorted(example_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")
print()
print("model_kind.txt :", (example_dir / "model_kind.txt").read_text())
print("params.json    :")
print((example_dir / "params.json").read_text())


## What's next

* **NB 05 — `05_training.ipynb`** — the `Trainer` that orchestrates `ModelFactory.create → fit → predict → score → wandb.log_artifact`, plus the `TrainedBundle` value object that pairs a fitted model with the `FeatureSpec` it consumes (so inference reconstructs identical transforms from a single artifact).
* **NB 06** — proper validation splitters (random / temporal / station-LOSO / spatial-block) and the metrics class.

The discipline going forward, mirroring NB 02 / 03:

| Change | What ripples |
|---|---|
| New hyperparameters on RF | New `RandomForestParams` instance → train → bump model artifact version |
| Add a fourth model class (e.g. XGBoost) | New `ModelKind` member + params dataclass + concrete subclass + entries in `ModelKind.params_type()` / `model_class()`. Three lines per file. |
| Switch a model from no-scaling to scaling | New params field; the wrapper's `fit/predict/save/load` already accommodates the change |

The model layer is single-responsibility: it consumes `(X, y)` and produces predictions. It deliberately knows nothing about W&B, MLflow, or the warehouse. Wiring those concerns belongs to the trainer (NB 05).
